In [3]:
import os
import numpy as np
import tomopy as tp # standard reconstruction algorithms
import wget        # library for URL-downloads
import dxchange    # library fpr opening DX-files (tomographic format)

import matplotlib.pyplot as plt

# Образ преобразования Радона

В этом задании вам предлагаются две практические задачи, решение которых оказывается очень удачно опирается на структуру образа преобразования Радона, которая описывается полностью следующей теоремой:

**Теорема. (Гельфанд-Граев-Хелгасон)** Пусть $f\in \mathcal{S}(R^n)$. Тогда $g(s,\theta) = Rf(s,\theta)$ удовлетворяет следующим свойствам:
1. $g\in\mathcal{S}(Z)$
2. $g(s,\theta) = g(-s, -\theta)$
3. для любого $k = 0, 1, \dots$ функции $I_k(\theta)$
\begin{equation}
    I_k(\theta) = \int\limits_{-\infty}^{+\infty}s^k Rf(s,\theta)\, ds, \, \theta \in S^{n-1}.
\end{equation}
являются ограничением однородных полиномов степени $k$ от $\theta_1, \theta_2, \dots, \theta_n$ на сферу $S^{n-1}$. Также если для фуннкции $g(s,\theta)$ выполняются условия (1)-(3), то найдется $f\in \mathcal{S}(R^n)$, такая что $g=Rf$.

## 1. Микротомография (MicroCT) (0.5 балла)

Рентгеновская микротомография (X-ray microtomography, MicroCT) - ещё один раздел рентгеновской томографии, занимающийся
восстановлением изображений структуры маленьких объектов (обычно до нескольких миллиметров в диаметре). 


<table>
<tr>
 <td> <img src="./microct-1.png" alt="micro-ct-1" style="width: 400px;"/> </td>
 <td> <img src="./microct-2.jpg" alt="micro-ct-2" style="width: 300px;"/> </td>
</tr>
<tr>
    <td> снимок радиолярии полученный при помощи MicroCT</td>
    <td> сечение малоберцовой кости мыши</td>
</tr>
</table>



По физической модели и использовании преобразований Радона, микротомография ничем не отличается от обычной рентгеновской. Но есть несколько важных практических отличий:

  * как правило - мишени неживые (если только это не маленькие животные), то есть нет ограничений по 
    дозе облучения, а значит и проблем со статистическим шумом
  * используемый пучок рентгеновских фотонов получают при помощи синхротрона с помощью которого можно гарантировать черезвычайно малую расходимость пучка и более того - оптическую когерентность фотонов (сохранение фазы)
  * излучатель и детектор не вращаются вокруг объекта, вместо этого объект закрепляется на стойке и вращается 
    вокруг своей оси; благодаря этому нет ограничения по шагу дискретизации в лучевых данных
    
Вам предлагается сделать восстановление в задаче микротомографии, где облучаемый объект - человеческий зуб. 

**Задача 1.1** Ввиду такого маленького разрешения возникает следующая проблема: центр вращения объекта на самом деле неизвестен, установить вручную его с точностью порядка разрешающей способности очень тяжело технически. В итоге лучевые данные искажены - центр вращения объекта не соответствует координате 0 по переменной $s$ (см. рисунок). Вам необходимо скорректировать данные и получить изображение сечения зуба (на нём должны быть видны полости и трещины).

Код ниже загружает лучевые данные и переводит в формат `numpy.ndarray`. Ваша задача заключается в том, чтобы восстановить снимок зуба по этим лучевым данным - на нём должна быть видна внутренняя структура зуба и прочие мелкие детали.

In [ ]:
# download h5 data with counts data

fname = 'tooth.h5'
data_status = os.path.exists('./' + fname)

if data_status == False:
    url = 'https://raw.github.com/fedor-goncharov/pdo-tomography-course/master/seminar-materials/seminar-3/tooth.h5'
    output_path = './'
    wget.download(url, output_path)

In [ ]:
# 1. preprocess data 
proj, flat, dark, theta = dxchange.read_aps_32id(
    fname = fname,
    sino = (0,2),
    )

# 1.1 normalize projection data and take -log
proj = tp.normalize(proj, flat, dark)
proj = tp.minus_log(proj)

# 1.2 convert to numpy.ndarray of size 181 x 640
proj = np.reshape(proj[:, 0, :], (181, 640))

# 1.3 (optional) plot image of raw data
plt.imshow(proj)
plt.title('MicroCT raw data')
plt.show()

In [8]:
# YOUR CODE HERE

## 2. Формулы Гончарова в электронной микроскопии (0.5 балла)

В электронной микроскопии исследуемые объекты имеют размеры существенно меньше чем в случае MicroCT (порядка микрометров и меньше). Облучение объекта идет пучком электронов, тем не менее, в некоторых моделях исходные данные можно моделировать классическими преобразованиями Радона. 

<img src="./microscope.png" alt="micro-ct-1" style="width: 400px;"/>

Возникает следующая практическая проблема:
 * исследуемый объект невозможно "вращать" ввиду его малости (например бактерия или клетка). Можно считать, что  объект находится в некоторой среде и случайно поворачивается, поэтому в различные времена микроскоп получает данные для разных направлений $Rf(s,\theta_i), \, \theta_1, \theta_2, \dots, \theta_n$, другое дело, что сами $\theta_i$ - **неизвестны**. 
 
Поэтому:
 1. Из лучевых  данных $Rf(s,\theta_i), \, \theta_1, \theta_2, \dots, \theta_n$ требуется восстановить направления $\theta_i$ (хотя бы с точностью до общего поворота). 
 2. Восстановить объект по лучевым данным $Rf(s,\theta_i), \, \theta_1, \theta_2, \dots, \theta_n$, используя подходящий алгоритм восстановления. 
 
Алгоритм восстановления $\{\theta_i\}_{i=1}^n$ следующий: 

1. $\theta_i = (\cos\varphi, \sin\varphi), \, \varphi \in [0, \pi)$, поэтому достаточно восстановить $\{\varphi_i\}_{i=1}^n$ с точностью до общего поворота. Определим моменты 
\begin{equation}
    I_k(\varphi) = \int\limits_{-\infty}^{+\infty}s^k Rf(s,\theta(\varphi))\, ds, \, k = 0, 1, \dots
\end{equation}
2. Несложно показать, что моменты $I_1(\varphi)$, $I_2(\varphi)$ можно представить в следующем виде (можете доказать, если это неочевидно)
\begin{equation}
    I_1(\varphi) = a \cos(\varphi - \alpha), \, I_2(\varphi) = b\cos(2(\varphi-\beta)) + c.
\end{equation}
Рассмотрим кривую $\Gamma$ на плоскости $R^2$ которая задаётся парметрически следующим образом:
\begin{align}
    \Gamma = \{x(\varphi), y(\varphi) | x(\varphi) = I_1(\varphi), \, y(\varphi) = I_2(\varphi), \, \varphi \in [0, 2\pi] \}
\end{align}

**Определение 1.** Кривая $\Gamma$ на $R^2$ называется алгебраической порядка $k$, если множество её точек $(x,y)\in \Gamma$ представимо в виде решения уравнения 
\begin{equation}
    P(x,y) = 0,
\end{equation}
где 
\begin{equation}
    P \text{ - однородный полином от двух переменных степени }k. 
\end{equation}

**Лемма 1.** Кривая $\Gamma$ определённая выше является алгебраической кривой степени 4. 
 
3. Однородный полином степени $4$ определяется своими $15$-ую коэффициентами. Восстановите кривую $\Gamma$ (коэффициенты соответствующего полинома) из лучевых данных (даже если мы и не знаем углы), если учесть, что лучевые данные содержат 15 направлений.

4. Чтобы восстановить a, b, c, которые входят в параметризацию $I_1(\varphi)$, $I_2(\varphi)$ воспользуемся формулами (проверьте их)
\begin{equation}
    a = (x_{max} - x_{min})/2, \, b = (y_{max} - y_{min})/2, \, c = (y_{max} + y_{min})/2, 
\end{equation}
где $x_{max}, x_{min}$, $y_{min}$, $y_{max}$, точки принадлежащие кривой $\Gamma$.

5. Из условия, что $\{\varphi_{i}\}_{i=1}^n$ необходимо восстановить до общего поворота, будем считать, что 
\begin{equation}
    \varphi_1 = 0.
\end{equation}
Из набора данных $\cos(\alpha), \, \cos(2\beta), \cos(\varphi_i-\alpha), \, \cos(2(\varphi_i - \beta))$ можно восстановить все углы $\varphi_i$, $i = 1, 2, \dots, 15$.


**Задача 2.1** Используйте алгоритм выше для того, чтобы найти направления 15-и неизвестных проекций. Код ниже загружаен лучевые данные с сервера.

In [375]:
# download h5 data with counts data

fname = 'microscope-data.bin'
data_status = os.path.exists('./' + fname)

if data_status == False:
    url = 'https://raw.github.com/fedor-goncharov/pdo-tomography-course/master/seminar-materials/seminar-3/' + fname
    output_path = './'
    wget.download(url, output_path)

In [ ]:
nshift = 1024
ntheta = 150

projections = np.reshape(np.fromfile(fname, dtype=np.float), (ntheta, nshift))
plt.imshow(projections) # show unordered projection data

## Итеративный метод ART для произвольной геометрии сканирования


Пусть необходимо решить линейную систему:
\begin{equation}
    Af = g, \, A\in \mathrm{Mat}(d, p), \, f\in R^p, \, g\in R^d.
\end{equation}

Пусть также $f_k$ - текущая аппроксимация $f$. Тогда обновить $f_k$ можно ортогонально спроецировав его на афинную плоскость 

\begin{equation}
    a_i^Tf = g_i, \, i\in \{1, \dots, d\}.
\end{equation}

Обозначим через $P_i$ - оператор ортогональной проекции на гиперплоскость $a_i^Tf = g_i$. Тогда алгоритм ART можно записать через следующий псевдокод:

```
    Algorithm ART
    
    set k=0, f(0) be the initial point
    repeat until convergence (set stopping criterion)
    
        u(0) := f(k)
        for n in (0, ..., d-1) do
            u(n+1) := Pn * u(n)
        endfor     
        f(k+1) := u(d)
        k := k + 1
        
    endrepeat
    
```


**Задача 2.2** Используйте алгоритм ART (или любой другой, подходящий для решения данной линейной сиситемы) для того чтобы из полученных данных восстановить исходное изображение. *Подсказка:* для работы алгоритма ART вам необходимо уметь вычислять строку матрицы преобразования Радона - для этого можете использовать код из первой лекции, либо код предоставленный лектором `line_projector_siddon`, который интегрирует точно изображение вдоль линии. Также вам необходимо самостоятельно найти форму проектора $P_i$. Можно также использовать стандартную функцию `iradon_sart` из библиотеки `scikit.image` (см. документацию).

**Параметры:** исходное изображение имело размер 128x128,  150 случайных проекций, 1024 луча в каждой проекции равномерно распределённых на отрезке [-1, 1]

In [ ]:
# YOUR CODE HERe